In [2]:
from pathlib import Path
import random

# =========================================================
# CONFIG
# =========================================================

# Root folder containing:
#   nutrition5k_compactV3/
#   splits/
DATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset")

COMPACT_DIR = DATASET_ROOT / "nutrition5k_compactV3"
SPLITS_DIR = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\dish_ids\splits")

# Choose which official split should define the experiment:
# "rgb"   -> based on overhead rgb split
# "depth" -> based on depth split
BASE_SPLIT = "rgb"

# Which validation percentage should be carved out of the AVAILABLE TOTAL dataset
TARGET_VAL_PCT_OF_TOTAL = 0.10

# Random seed for reproducibility
SEED = 42

# Output folder
OUTPUT_DIR = DATASET_ROOT / f"splits_compact_{BASE_SPLIT}_with_val"

In [3]:
from pathlib import Path
import random


# =========================================================
# HELPERS
# =========================================================

def read_ids(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def write_ids(path: Path, ids):
    with open(path, "w", encoding="utf-8") as f:
        for dish_id in ids:
            f.write(f"{dish_id}\n")

def pct(n, total):
    return 100 * n / total if total else 0.0

def discover_available_dishes(compact_dir: Path):
    """
    Scans the compact dataset and records which dishes have which files.
    """
    dishes = {}

    for dish_dir in compact_dir.iterdir():
        if not dish_dir.is_dir():
            continue
        if not dish_dir.name.startswith("dish_"):
            continue

        overhead_dir = dish_dir / "overhead"

        rgb_exists = (overhead_dir / "rgb.png").exists()
        depth_raw_exists = (overhead_dir / "depth_raw.png").exists()
        depth_color_exists = (overhead_dir / "depth_color.png").exists()

        dishes[dish_dir.name] = {
            "rgb": rgb_exists,
            "depth": depth_raw_exists or depth_color_exists,
            "depth_raw": depth_raw_exists,
            "depth_color": depth_color_exists,
        }

    return dishes

def create_val_split(train_ids, desired_val_count, seed=42):
    ids = list(train_ids)
    rng = random.Random(seed)
    rng.shuffle(ids)

    val_ids = sorted(ids[:desired_val_count])
    train_ids = sorted(ids[desired_val_count:])
    return train_ids, val_ids

In [4]:
# =========================================================
# MAIN
# =========================================================

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rgb_train_ids = set(read_ids(SPLITS_DIR / "rgb_train_ids.txt"))
    rgb_test_ids  = set(read_ids(SPLITS_DIR / "rgb_test_ids.txt"))
    depth_train_ids = set(read_ids(SPLITS_DIR / "depth_train_ids.txt"))
    depth_test_ids  = set(read_ids(SPLITS_DIR / "depth_test_ids.txt"))

    # Sanity checks
    assert len(rgb_train_ids & rgb_test_ids) == 0, "RGB train/test overlap detected"
    assert len(depth_train_ids & depth_test_ids) == 0, "Depth train/test overlap detected"

    dishes = discover_available_dishes(COMPACT_DIR)
    all_available = set(dishes.keys())

    available_rgb_dishes = {dish_id for dish_id, info in dishes.items() if info["rgb"]}
    available_depth_dishes = {dish_id for dish_id, info in dishes.items() if info["depth"]}

    print("=" * 70)
    print("AVAILABLE DISHES IN MODIFIED DATASET")
    print("=" * 70)
    print(f"All dish folders found:                 {len(all_available)}")
    print(f"Dishes with overhead rgb.png:          {len(available_rgb_dishes)}")
    print(f"Dishes with overhead depth file(s):    {len(available_depth_dishes)}")
    print()

    # -----------------------------------------------------
    # Audit RGB split against modified dataset
    # -----------------------------------------------------
    rgb_train_available = sorted(rgb_train_ids & available_rgb_dishes)
    rgb_test_available  = sorted(rgb_test_ids & available_rgb_dishes)
    rgb_total_available = len(rgb_train_available) + len(rgb_test_available)

    print("=" * 70)
    print("OFFICIAL RGB SPLIT AFTER INTERSECTION")
    print("=" * 70)
    print(f"RGB train available: {len(rgb_train_available)}")
    print(f"RGB test available:  {len(rgb_test_available)}")
    print(f"RGB total available: {rgb_total_available}")
    if rgb_total_available > 0:
        print(f"RGB train %:         {pct(len(rgb_train_available), rgb_total_available):.2f}%")
        print(f"RGB test %:          {pct(len(rgb_test_available), rgb_total_available):.2f}%")
    print()

    # -----------------------------------------------------
    # Audit depth split against modified dataset
    # -----------------------------------------------------
    depth_train_available = sorted(depth_train_ids & available_depth_dishes)
    depth_test_available  = sorted(depth_test_ids & available_depth_dishes)
    depth_total_available = len(depth_train_available) + len(depth_test_available)

    print("=" * 70)
    print("OFFICIAL DEPTH SPLIT AFTER INTERSECTION")
    print("=" * 70)
    print(f"Depth train available: {len(depth_train_available)}")
    print(f"Depth test available:  {len(depth_test_available)}")
    print(f"Depth total available: {depth_total_available}")
    if depth_total_available > 0:
        print(f"Depth train %:         {pct(len(depth_train_available), depth_total_available):.2f}%")
        print(f"Depth test %:          {pct(len(depth_test_available), depth_total_available):.2f}%")
    print()

    # -----------------------------------------------------
    # Select base split
    # -----------------------------------------------------
    if BASE_SPLIT == "rgb":
        base_train = rgb_train_available
        base_test = rgb_test_available
        base_total = rgb_total_available
        required_modality = "rgb"
    elif BASE_SPLIT == "depth":
        base_train = depth_train_available
        base_test = depth_test_available
        base_total = depth_total_available
        required_modality = "depth"
    else:
        raise ValueError("BASE_SPLIT must be either 'rgb' or 'depth'")

    if base_total == 0:
        raise ValueError(f"No dishes available for BASE_SPLIT='{BASE_SPLIT}'")

    # Desired validation count as 10% of total available dataset
    desired_val_count = round(TARGET_VAL_PCT_OF_TOTAL * base_total)

    if desired_val_count >= len(base_train):
        raise ValueError(
            "Desired validation split is too large relative to available training dishes."
        )

    final_train, final_val = create_val_split(
        train_ids=base_train,
        desired_val_count=desired_val_count,
        seed=SEED
    )
    final_test = sorted(base_test)

    # Final safety checks
    s_train = set(final_train)
    s_val = set(final_val)
    s_test = set(final_test)

    assert len(s_train & s_val) == 0, "Train/val overlap"
    assert len(s_train & s_test) == 0, "Train/test overlap"
    assert len(s_val & s_test) == 0, "Val/test overlap"

    union_total = len(s_train | s_val | s_test)
    assert union_total == base_total, "Final split does not cover all selected dishes"

    # Save split files
    write_ids(OUTPUT_DIR / "train_ids.txt", final_train)
    write_ids(OUTPUT_DIR / "val_ids.txt", final_val)
    write_ids(OUTPUT_DIR / "test_ids.txt", final_test)

    # Save an audit report
    report = []
    report.append(f"BASE_SPLIT = {BASE_SPLIT}")
    report.append(f"Required modality = {required_modality}")
    report.append(f"Random seed = {SEED}")
    report.append("")

    report.append("Available dishes in modified dataset")
    report.append(f"All dish folders: {len(all_available)}")
    report.append(f"RGB available: {len(available_rgb_dishes)}")
    report.append(f"Depth available: {len(available_depth_dishes)}")
    report.append("")

    report.append("Official RGB split after intersection")
    report.append(f"Train: {len(rgb_train_available)}")
    report.append(f"Test:  {len(rgb_test_available)}")
    report.append(f"Total: {rgb_total_available}")
    if rgb_total_available > 0:
        report.append(f"Train %: {pct(len(rgb_train_available), rgb_total_available):.2f}%")
        report.append(f"Test %:  {pct(len(rgb_test_available), rgb_total_available):.2f}%")
    report.append("")

    report.append("Official depth split after intersection")
    report.append(f"Train: {len(depth_train_available)}")
    report.append(f"Test:  {len(depth_test_available)}")
    report.append(f"Total: {depth_total_available}")
    if depth_total_available > 0:
        report.append(f"Train %: {pct(len(depth_train_available), depth_total_available):.2f}%")
        report.append(f"Test %:  {pct(len(depth_test_available), depth_total_available):.2f}%")
    report.append("")

    report.append("Final selected split with validation")
    report.append(f"Train: {len(final_train)} ({pct(len(final_train), base_total):.2f}%)")
    report.append(f"Val:   {len(final_val)} ({pct(len(final_val), base_total):.2f}%)")
    report.append(f"Test:  {len(final_test)} ({pct(len(final_test), base_total):.2f}%)")

    with open(OUTPUT_DIR / "split_report.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(report))

    print("=" * 70)
    print(f"FINAL SPLIT CREATED USING BASE_SPLIT = {BASE_SPLIT}")
    print("=" * 70)
    print(f"Train: {len(final_train)} ({pct(len(final_train), base_total):.2f}%)")
    print(f"Val:   {len(final_val)} ({pct(len(final_val), base_total):.2f}%)")
    print(f"Test:  {len(final_test)} ({pct(len(final_test), base_total):.2f}%)")
    print()
    print(f"Saved files to: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

AVAILABLE DISHES IN MODIFIED DATASET
All dish folders found:                 3493
Dishes with overhead rgb.png:          3490
Dishes with overhead depth file(s):    3490

OFFICIAL RGB SPLIT AFTER INTERSECTION
RGB train available: 2755
RGB test available:  507
RGB total available: 3262
RGB train %:         84.46%
RGB test %:          15.54%

OFFICIAL DEPTH SPLIT AFTER INTERSECTION
Depth train available: 2755
Depth test available:  507
Depth total available: 3262
Depth train %:         84.46%
Depth test %:          15.54%

FINAL SPLIT CREATED USING BASE_SPLIT = rgb
Train: 2429 (74.46%)
Val:   326 (9.99%)
Test:  507 (15.54%)

Saved files to: C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\splits_compact_rgb_with_val


It seems that from the original splits in the dataset they only work with data from dish_ids_cafe1 and not dish_ids_cafe2 this will be tested below if this was the cause of the mismatch. If so we will go ahead with the original splitting of the dataset.

In [5]:
from pathlib import Path

DATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset")
COMPACT_DIR = DATASET_ROOT / "nutrition5k_compactV3"
SPLITS_DIR = SPLITS_DIR = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\dish_ids\splits")

def read_ids(path):
    with open(path, "r", encoding="utf-8") as f:
        return set(line.strip() for line in f if line.strip())

all_compact = {
    p.name for p in COMPACT_DIR.iterdir()
    if p.is_dir() and p.name.startswith("dish_")
}

rgb_train = read_ids(SPLITS_DIR / "rgb_train_ids.txt")
rgb_test = read_ids(SPLITS_DIR / "rgb_test_ids.txt")
depth_train = read_ids(SPLITS_DIR / "depth_train_ids.txt")
depth_test = read_ids(SPLITS_DIR / "depth_test_ids.txt")

official_all = rgb_train | rgb_test
unmatched = sorted(all_compact - official_all)

print("Compact dishes:", len(all_compact))
print("Official split dishes present:", len(all_compact & official_all))
print("Unmatched dishes:", len(unmatched))
print("\nFirst 30 unmatched:")
for x in unmatched[:30]:
    print(x)

Compact dishes: 3493
Official split dishes present: 3265
Unmatched dishes: 228

First 30 unmatched:
dish_1571931457
dish_1571931482
dish_1571931594
dish_1571931648
dish_1571931678
dish_1571931763
dish_1571931782
dish_1571932341
dish_1571932375
dish_1571932398
dish_1571932448
dish_1571934124
dish_1571934154
dish_1571934465
dish_1571934494
dish_1571935085
dish_1571935895
dish_1571944214
dish_1571944238
dish_1571944487
dish_1571944507
dish_1571944531
dish_1571944549
dish_1571944571
dish_1571944648
dish_1571945939
dish_1571946018
dish_1571946063
dish_1571946265
dish_1571946303


In [6]:
from pathlib import Path

# =========================================================
# CONFIG
# =========================================================

DATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset")
OGDATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\dish_ids")

COMPACT_DIR = DATASET_ROOT / "nutrition5k_compactV3"
SPLITS_DIR = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\dish_ids\splits")

CAFE1_FILE = OGDATASET_ROOT / "dish_ids_cafe1.txt"
CAFE2_FILE = OGDATASET_ROOT / "dish_ids_cafe2.txt"

OUTPUT_DIR = DATASET_ROOT / "split_audit_unmatched"

# =========================================================
# HELPERS
# =========================================================

def read_ids(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return set(line.strip() for line in f if line.strip())

def write_ids(path: Path, ids):
    with open(path, "w", encoding="utf-8") as f:
        for dish_id in sorted(ids):
            f.write(f"{dish_id}\n")

def discover_compact_dishes(compact_dir: Path):
    return {
        p.name for p in compact_dir.iterdir()
        if p.is_dir() and p.name.startswith("dish_")
    }

# =========================================================
# MAIN
# =========================================================

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Compact dataset dishes
    compact_dishes = discover_compact_dishes(COMPACT_DIR)

    # Official splits
    rgb_train = read_ids(SPLITS_DIR / "rgb_train_ids.txt")
    rgb_test = read_ids(SPLITS_DIR / "rgb_test_ids.txt")
    depth_train = read_ids(SPLITS_DIR / "depth_train_ids.txt")
    depth_test = read_ids(SPLITS_DIR / "depth_test_ids.txt")

    official_rgb_all = rgb_train | rgb_test
    official_depth_all = depth_train | depth_test

    # Cafe membership files
    cafe1_ids = read_ids(CAFE1_FILE)
    cafe2_ids = read_ids(CAFE2_FILE)

    # Unmatched against official split universes
    unmatched_rgb = compact_dishes - official_rgb_all
    unmatched_depth = compact_dishes - official_depth_all

    # Since your earlier output showed RGB/depth matched equally,
    # these will probably be the same, but we test both anyway.
    for label, unmatched in [
        ("rgb", unmatched_rgb),
        ("depth", unmatched_depth),
    ]:
        in_cafe1 = unmatched & cafe1_ids
        in_cafe2 = unmatched & cafe2_ids
        in_both = in_cafe1 & in_cafe2
        in_neither = unmatched - (cafe1_ids | cafe2_ids)

        print("=" * 70)
        print(f"UNMATCHED ANALYSIS AGAINST OFFICIAL {label.upper()} SPLIT")
        print("=" * 70)
        print(f"Compact dishes total:                    {len(compact_dishes)}")
        print(f"Official {label} split universe size:      {len(official_rgb_all) if label == 'rgb' else len(official_depth_all)}")
        print(f"Unmatched compact dishes:               {len(unmatched)}")
        print()
        print(f"Unmatched dishes found in cafe1 list:   {len(in_cafe1)}")
        print(f"Unmatched dishes found in cafe2 list:   {len(in_cafe2)}")
        print(f"Unmatched dishes found in BOTH:         {len(in_both)}")
        print(f"Unmatched dishes in NEITHER list:       {len(in_neither)}")
        print()

        if len(unmatched) > 0:
            print("Percent breakdown of unmatched dishes:")
            print(f"  cafe1:   {100 * len(in_cafe1) / len(unmatched):.2f}%")
            print(f"  cafe2:   {100 * len(in_cafe2) / len(unmatched):.2f}%")
            print(f"  neither: {100 * len(in_neither) / len(unmatched):.2f}%")
        print()

        print("First 20 unmatched dish IDs:")
        for dish_id in sorted(unmatched)[:20]:
            print(f"  {dish_id}")
        print()

        # Save files
        write_ids(OUTPUT_DIR / f"unmatched_{label}.txt", unmatched)
        write_ids(OUTPUT_DIR / f"unmatched_{label}_in_cafe1.txt", in_cafe1)
        write_ids(OUTPUT_DIR / f"unmatched_{label}_in_cafe2.txt", in_cafe2)
        write_ids(OUTPUT_DIR / f"unmatched_{label}_in_neither.txt", in_neither)

    # Also compare official split universes directly to cafe sets
    print("=" * 70)
    print("OFFICIAL SPLIT COVERAGE OF CAFE SETS")
    print("=" * 70)

    rgb_cafe1 = official_rgb_all & cafe1_ids
    rgb_cafe2 = official_rgb_all & cafe2_ids
    depth_cafe1 = official_depth_all & cafe1_ids
    depth_cafe2 = official_depth_all & cafe2_ids

    print(f"RGB split dishes in cafe1:    {len(rgb_cafe1)}")
    print(f"RGB split dishes in cafe2:    {len(rgb_cafe2)}")
    print(f"Depth split dishes in cafe1:  {len(depth_cafe1)}")
    print(f"Depth split dishes in cafe2:  {len(depth_cafe2)}")
    print()

    print(f"Total cafe1 dishes:           {len(cafe1_ids)}")
    print(f"Total cafe2 dishes:           {len(cafe2_ids)}")

if __name__ == "__main__":
    main()

UNMATCHED ANALYSIS AGAINST OFFICIAL RGB SPLIT
Compact dishes total:                    3493
Official rgb split universe size:      4768
Unmatched compact dishes:               228

Unmatched dishes found in cafe1 list:   0
Unmatched dishes found in cafe2 list:   228
Unmatched dishes found in BOTH:         0
Unmatched dishes in NEITHER list:       0

Percent breakdown of unmatched dishes:
  cafe1:   0.00%
  cafe2:   100.00%
  neither: 0.00%

First 20 unmatched dish IDs:
  dish_1571931457
  dish_1571931482
  dish_1571931594
  dish_1571931648
  dish_1571931678
  dish_1571931763
  dish_1571931782
  dish_1571932341
  dish_1571932375
  dish_1571932398
  dish_1571932448
  dish_1571934124
  dish_1571934154
  dish_1571934465
  dish_1571934494
  dish_1571935085
  dish_1571935895
  dish_1571944214
  dish_1571944238
  dish_1571944487

UNMATCHED ANALYSIS AGAINST OFFICIAL DEPTH SPLIT
Compact dishes total:                    3493
Official depth split universe size:      3265
Unmatched compact dishes:

We have now found the root cause of the problem, the next code block will run the script once again, and create the split (training / validation / testing) along with a csv file containing only the dish id, and the total_carb amount. This will make it easier to work with in colab. It seems that the dish_ids are created through a unix time code stamp, so to ensure that there isnt any data leakage between the traning and validation splits this will be accounted for by adding them into buckets before splitting them. The splitting will also be tested in a subsequent code block which tests for similarities between the two splits.

In [10]:
from pathlib import Path
from collections import defaultdict
import random
import csv
import pandas as pd

# =========================================================
# CONFIG
# =========================================================

DATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset")
OGDATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\dish_ids")
OGDATASET_ROOT2 = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\metadata")

COMPACT_DIR = DATASET_ROOT / "nutrition5k_compactV3"
SPLITS_DIR = OGDATASET_ROOT / "splits"

DISH_METADATA_CAFE1 = OGDATASET_ROOT2 / "dish_metadata_cafe1.csv"
DISH_METADATA_CAFE2 = OGDATASET_ROOT2 / "dish_metadata_cafe2.csv"

OUTPUT_DIR = DATASET_ROOT / "prepared_overhead_rgb_dataset"

# Use official split universe as the benchmark basis
BASE_SPLIT = "depth"   # "depth" recommended, "rgb" also possible

# Create validation from official training only
TARGET_VAL_PCT_OF_TOTAL = 0.10
SEED = 42

# Split mode:
#   "random"      = random split within official training
#   "time_bucket" = group-aware proxy split based on dish timestamp
SPLIT_MODE = "time_bucket"

# Used only if SPLIT_MODE == "time_bucket"
BUCKET_SECONDS = 6 * 60 * 60   # 6 hours

# Which file to use only for checking that a dish exists in compact dataset
REQUIRED_REL_PATH = Path("overhead") / "rgb.png"

# =========================================================
# HELPERS
# =========================================================

def read_ids(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def write_ids(path: Path, ids):
    with open(path, "w", encoding="utf-8") as f:
        for dish_id in sorted(ids):
            f.write(f"{dish_id}\n")

def pct(n, total):
    return 100 * n / total if total else 0.0

def discover_available_dishes(compact_dir: Path, required_rel_path: Path):
    available = set()

    if not compact_dir.exists():
        raise FileNotFoundError(f"Compact dataset folder not found: {compact_dir}")

    for dish_dir in compact_dir.iterdir():
        if not dish_dir.is_dir():
            continue
        if not dish_dir.name.startswith("dish_"):
            continue
        if (dish_dir / required_rel_path).exists():
            available.add(dish_dir.name)

    return available

def dish_to_timestamp(dish_id: str):
    return int(dish_id.replace("dish_", ""))

def assign_time_bucket(dish_id: str, bucket_seconds: int):
    return dish_to_timestamp(dish_id) // bucket_seconds

def random_split(train_ids, target_val_count, seed=42):
    ids = list(train_ids)
    rng = random.Random(seed)
    rng.shuffle(ids)
    val_ids = sorted(ids[:target_val_count])
    train_ids = sorted(ids[target_val_count:])
    return train_ids, val_ids

def grouped_time_split(train_ids, target_val_count, bucket_seconds, seed=42):
    bucket_to_ids = defaultdict(list)
    for dish_id in train_ids:
        bucket = assign_time_bucket(dish_id, bucket_seconds)
        bucket_to_ids[bucket].append(dish_id)

    bucket_items = list(bucket_to_ids.items())
    rng = random.Random(seed)
    rng.shuffle(bucket_items)

    val_ids = []
    train_out = []
    current_val = 0

    for bucket, ids in bucket_items:
        if current_val < target_val_count:
            val_ids.extend(ids)
            current_val += len(ids)
        else:
            train_out.extend(ids)

    return sorted(train_out), sorted(val_ids)

def load_single_dish_metadata(path: Path, source_cafe: str):
    rows = []

    if not path.exists():
        raise FileNotFoundError(f"Metadata file not found: {path}")

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)

        for line_num, row in enumerate(reader, start=1):
            if not row:
                continue

            # Expected fixed leading columns:
            # 0=dish_id, 1=total_calories, 2=total_mass,
            # 3=total_fat, 4=total_carb, 5=total_protein
            if len(row) < 6:
                print(f"WARNING: Skipping short row in {path.name} at line {line_num}")
                continue

            dish_id = row[0].strip()

            try:
                total_carb = float(row[4])
            except ValueError:
                print(f"WARNING: Skipping unparsable row in {path.name} at line {line_num}")
                continue

            rows.append({
                "dish_id": dish_id,
                "total_carb": total_carb,
                "source_cafe": source_cafe
            })

    return pd.DataFrame(rows)

def load_dish_metadata():
    df1 = load_single_dish_metadata(DISH_METADATA_CAFE1, "cafe1")
    df2 = load_single_dish_metadata(DISH_METADATA_CAFE2, "cafe2")

    df = pd.concat([df1, df2], ignore_index=True)

    df["dish_id"] = df["dish_id"].astype(str).str.strip()

    dup_count = df["dish_id"].duplicated().sum()
    if dup_count > 0:
        print(f"WARNING: Found {dup_count} duplicate dish_id rows in metadata. Keeping first occurrence.")
        df = df.drop_duplicates(subset=["dish_id"], keep="first").reset_index(drop=True)

    return df

# =========================================================
# MAIN
# =========================================================

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # -----------------------------
    # Read official split files
    # -----------------------------
    if BASE_SPLIT == "rgb":
        official_train = set(read_ids(SPLITS_DIR / "rgb_train_ids.txt"))
        official_test  = set(read_ids(SPLITS_DIR / "rgb_test_ids.txt"))
    elif BASE_SPLIT == "depth":
        official_train = set(read_ids(SPLITS_DIR / "depth_train_ids.txt"))
        official_test  = set(read_ids(SPLITS_DIR / "depth_test_ids.txt"))
    else:
        raise ValueError("BASE_SPLIT must be 'rgb' or 'depth'")

    if official_train & official_test:
        raise ValueError("Official train/test overlap detected")

    # -----------------------------
    # Available dishes in compact dataset
    # -----------------------------
    available_dishes = discover_available_dishes(COMPACT_DIR, REQUIRED_REL_PATH)

    compact_train = sorted(official_train & available_dishes)
    compact_test  = sorted(official_test & available_dishes)

    total = len(compact_train) + len(compact_test)
    if total == 0:
        raise ValueError("No matched dishes found after filtering")

    print("=" * 70)
    print("MATCHED DATASET SUMMARY")
    print("=" * 70)
    print(f"Available dishes with required file: {len(available_dishes)}")
    print(f"Matched official train dishes:      {len(compact_train)}")
    print(f"Matched official test dishes:       {len(compact_test)}")
    print(f"Matched total dishes:               {total}")
    print()

    # -----------------------------
    # Create validation set
    # -----------------------------
    target_val_count = round(TARGET_VAL_PCT_OF_TOTAL * total)
    if target_val_count >= len(compact_train):
        raise ValueError("Validation target too large for available training set")

    if SPLIT_MODE == "random":
        final_train, final_val = random_split(compact_train, target_val_count, seed=SEED)
    elif SPLIT_MODE == "time_bucket":
        final_train, final_val = grouped_time_split(
            compact_train,
            target_val_count=target_val_count,
            bucket_seconds=BUCKET_SECONDS,
            seed=SEED
        )
    else:
        raise ValueError("SPLIT_MODE must be 'random' or 'time_bucket'")

    final_test = sorted(compact_test)

    # -----------------------------
    # Safety checks
    # -----------------------------
    s_train = set(final_train)
    s_val = set(final_val)
    s_test = set(final_test)

    if s_train & s_val:
        raise ValueError("Train/val overlap")
    if s_train & s_test:
        raise ValueError("Train/test overlap")
    if s_val & s_test:
        raise ValueError("Val/test overlap")

    if len(s_train | s_val | s_test) != total:
        raise ValueError("Final split does not cover all selected dishes")

    print("=" * 70)
    print("FINAL SPLIT SUMMARY")
    print("=" * 70)
    print(f"Train: {len(final_train)} ({pct(len(final_train), total):.2f}%)")
    print(f"Val:   {len(final_val)} ({pct(len(final_val), total):.2f}%)")
    print(f"Test:  {len(final_test)} ({pct(len(final_test), total):.2f}%)")
    print()

    # -----------------------------
    # Load metadata
    # -----------------------------
    meta = load_dish_metadata()
    meta = meta[["dish_id", "total_carb", "source_cafe"]].copy()

    split_ids = s_train | s_val | s_test

    meta_before_filter = len(meta)
    meta = meta[meta["dish_id"].isin(split_ids)].copy()
    meta_after_filter = len(meta)

    missing_in_metadata = sorted(split_ids - set(meta["dish_id"]))

    print("=" * 70)
    print("METADATA MATCHING SUMMARY")
    print("=" * 70)
    print(f"Metadata rows before split filter: {meta_before_filter}")
    print(f"Metadata rows after split filter:  {meta_after_filter}")
    print(f"Split dishes missing in metadata:  {len(missing_in_metadata)}")
    if missing_in_metadata:
        print("First 20 missing in metadata:")
        for dish_id in missing_in_metadata[:20]:
            print(f"  {dish_id}")
    print()

    # Add split column
    split_map = {}
    for x in final_train:
        split_map[x] = "train"
    for x in final_val:
        split_map[x] = "val"
    for x in final_test:
        split_map[x] = "test"

    meta["split"] = meta["dish_id"].map(split_map)

    # Final metadata table
    dataset_df = meta[["dish_id", "total_carb", "split", "source_cafe"]].copy()
    dataset_df = dataset_df.sort_values(["split", "dish_id"]).reset_index(drop=True)

    csv_train = (dataset_df["split"] == "train").sum()
    csv_val = (dataset_df["split"] == "val").sum()
    csv_test = (dataset_df["split"] == "test").sum()

    # -----------------------------
    # Save outputs
    # -----------------------------
    write_ids(OUTPUT_DIR / "train_ids.txt", final_train)
    write_ids(OUTPUT_DIR / "val_ids.txt", final_val)
    write_ids(OUTPUT_DIR / "test_ids.txt", final_test)

    dataset_df.to_csv(OUTPUT_DIR / "dish_carb_split_metadata.csv", index=False)

    with open(OUTPUT_DIR / "split_report.txt", "w", encoding="utf-8") as f:
        f.write(f"BASE_SPLIT={BASE_SPLIT}\n")
        f.write(f"SPLIT_MODE={SPLIT_MODE}\n")
        f.write(f"SEED={SEED}\n")
        if SPLIT_MODE == "time_bucket":
            f.write(f"BUCKET_SECONDS={BUCKET_SECONDS}\n")
        f.write(f"REQUIRED_REL_PATH={REQUIRED_REL_PATH.as_posix()}\n")
        f.write(f"TOTAL_MATCHED={total}\n")
        f.write(f"TRAIN_IDS={len(final_train)} ({pct(len(final_train), total):.2f}%)\n")
        f.write(f"VAL_IDS={len(final_val)} ({pct(len(final_val), total):.2f}%)\n")
        f.write(f"TEST_IDS={len(final_test)} ({pct(len(final_test), total):.2f}%)\n")
        f.write(f"CSV_ROWS={len(dataset_df)}\n")
        f.write(f"CSV_TRAIN_ROWS={csv_train}\n")
        f.write(f"CSV_VAL_ROWS={csv_val}\n")
        f.write(f"CSV_TEST_ROWS={csv_test}\n")
        f.write(f"MISSING_IN_METADATA={len(missing_in_metadata)}\n")
        f.write(f"CAFE1_ROWS={(dataset_df['source_cafe'] == 'cafe1').sum()}\n")
        f.write(f"CAFE2_ROWS={(dataset_df['source_cafe'] == 'cafe2').sum()}\n")

    print("=" * 70)
    print("DATASET PREPARATION COMPLETE")
    print("=" * 70)
    print(f"Train IDs: {len(final_train)}")
    print(f"Val IDs:   {len(final_val)}")
    print(f"Test IDs:  {len(final_test)}")
    print()
    print(f"CSV rows:  {len(dataset_df)}")
    print(f"CSV train: {csv_train}")
    print(f"CSV val:   {csv_val}")
    print(f"CSV test:  {csv_test}")
    print()
    print(f"Saved split files and metadata CSV to:\n{OUTPUT_DIR}")

if __name__ == "__main__":
    main()

MATCHED DATASET SUMMARY
Available dishes with required file: 3490
Matched official train dishes:      2755
Matched official test dishes:       507
Matched total dishes:               3262

FINAL SPLIT SUMMARY
Train: 2421 (74.22%)
Val:   334 (10.24%)
Test:  507 (15.54%)

METADATA MATCHING SUMMARY
Metadata rows before split filter: 5006
Metadata rows after split filter:  3262
Split dishes missing in metadata:  0

DATASET PREPARATION COMPLETE
Train IDs: 2421
Val IDs:   334
Test IDs:  507

CSV rows:  3262
CSV train: 2421
CSV val:   334
CSV test:  507

Saved split files and metadata CSV to:
C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\prepared_overhead_rgb_dataset


Testing for correct training and validation split

In [11]:
from pathlib import Path
from collections import defaultdict
import csv
import statistics

# =========================================================
# CONFIG
# =========================================================

DATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset")
OGDATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\dish_ids")
OGDATASET_ROOT2 = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\metadata")

COMPACT_DIR = DATASET_ROOT / "nutrition5k_compactV3"
PREPARED_DIR = DATASET_ROOT / "prepared_overhead_rgb_dataset"

SPLITS_DIR = OGDATASET_ROOT / "splits"
DISH_METADATA_CAFE1 = OGDATASET_ROOT2 / "dish_metadata_cafe1.csv"
DISH_METADATA_CAFE2 = OGDATASET_ROOT2 / "dish_metadata_cafe2.csv"

TRAIN_FILE = PREPARED_DIR / "train_ids.txt"
VAL_FILE = PREPARED_DIR / "val_ids.txt"
TEST_FILE = PREPARED_DIR / "test_ids.txt"
METADATA_SPLIT_CSV = PREPARED_DIR / "dish_carb_split_metadata.csv"

BASE_SPLIT = "depth"     # "depth" or "rgb"
REQUIRED_REL_PATH = Path("overhead") / "rgb.png"

# Must match the split script if you used time-bucket splitting
BUCKET_SECONDS = 6 * 60 * 60

# =========================================================
# HELPERS
# =========================================================

def read_ids(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def read_csv_rows(path: Path):
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        return list(reader)

def discover_available_dishes(compact_dir: Path, required_rel_path: Path):
    available = set()
    for dish_dir in compact_dir.iterdir():
        if not dish_dir.is_dir():
            continue
        if not dish_dir.name.startswith("dish_"):
            continue
        if (dish_dir / required_rel_path).exists():
            available.add(dish_dir.name)
    return available

def dish_to_timestamp(dish_id: str):
    return int(dish_id.replace("dish_", ""))

def assign_time_bucket(dish_id: str, bucket_seconds: int):
    return dish_to_timestamp(dish_id) // bucket_seconds

def pct(n, total):
    return 100 * n / total if total else 0.0

def summarize_timestamps(ids):
    if not ids:
        return None
    ts = sorted(dish_to_timestamp(x) for x in ids)
    return {
        "min": min(ts),
        "max": max(ts),
        "median": int(statistics.median(ts))
    }

def print_header(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

# =========================================================
# METADATA LOADING
# =========================================================

def load_fixed_metadata_ids(path: Path):
    ids = set()
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            if len(row) < 1:
                continue
            ids.add(row[0].strip())
    return ids

# =========================================================
# MAIN
# =========================================================

def main():
    # -----------------------------
    # Input existence checks
    # -----------------------------
    required_files = [
        TRAIN_FILE, VAL_FILE, TEST_FILE, METADATA_SPLIT_CSV,
        DISH_METADATA_CAFE1, DISH_METADATA_CAFE2
    ]
    for p in required_files:
        if not p.exists():
            raise FileNotFoundError(f"Missing required file: {p}")

    # -----------------------------
    # Read generated split files
    # -----------------------------
    train_ids = read_ids(TRAIN_FILE)
    val_ids = read_ids(VAL_FILE)
    test_ids = read_ids(TEST_FILE)

    s_train = set(train_ids)
    s_val = set(val_ids)
    s_test = set(test_ids)

    # -----------------------------
    # Read official split files
    # -----------------------------
    if BASE_SPLIT == "rgb":
        official_train = set(read_ids(SPLITS_DIR / "rgb_train_ids.txt"))
        official_test = set(read_ids(SPLITS_DIR / "rgb_test_ids.txt"))
    elif BASE_SPLIT == "depth":
        official_train = set(read_ids(SPLITS_DIR / "depth_train_ids.txt"))
        official_test = set(read_ids(SPLITS_DIR / "depth_test_ids.txt"))
    else:
        raise ValueError("BASE_SPLIT must be 'rgb' or 'depth'")

    # -----------------------------
    # Read available dishes in compact dataset
    # -----------------------------
    available_dishes = discover_available_dishes(COMPACT_DIR, REQUIRED_REL_PATH)

    expected_train_pool = official_train & available_dishes
    expected_test_pool = official_test & available_dishes
    expected_total_pool = expected_train_pool | expected_test_pool

    # -----------------------------
    # Read metadata split CSV
    # -----------------------------
    csv_rows = read_csv_rows(METADATA_SPLIT_CSV)
    csv_ids = [row["dish_id"].strip() for row in csv_rows]
    csv_id_set = set(csv_ids)

    # -----------------------------
    # Read original cafe metadata IDs
    # -----------------------------
    cafe1_ids = load_fixed_metadata_ids(DISH_METADATA_CAFE1)
    cafe2_ids = load_fixed_metadata_ids(DISH_METADATA_CAFE2)

    # =====================================================
    # CHECKS
    # =====================================================

    print_header("BASIC SPLIT SUMMARY")
    total = len(s_train) + len(s_val) + len(s_test)
    print(f"Train IDs: {len(s_train)} ({pct(len(s_train), total):.2f}%)")
    print(f"Val IDs:   {len(s_val)} ({pct(len(s_val), total):.2f}%)")
    print(f"Test IDs:  {len(s_test)} ({pct(len(s_test), total):.2f}%)")
    print(f"Total IDs: {total}")

    print_header("OVERLAP CHECKS")
    print(f"train ∩ val  = {len(s_train & s_val)}")
    print(f"train ∩ test = {len(s_train & s_test)}")
    print(f"val ∩ test   = {len(s_val & s_test)}")

    print_header("DUPLICATE CHECKS INSIDE EACH FILE")
    print(f"Duplicate train IDs: {len(train_ids) - len(s_train)}")
    print(f"Duplicate val IDs:   {len(val_ids) - len(s_val)}")
    print(f"Duplicate test IDs:  {len(test_ids) - len(s_test)}")

    print_header("EXPECTED SPLIT UNIVERSE CHECK")
    generated_union = s_train | s_val | s_test
    missing_from_generated = expected_total_pool - generated_union
    extra_in_generated = generated_union - expected_total_pool

    print(f"Expected official+available universe: {len(expected_total_pool)}")
    print(f"Generated split union:                {len(generated_union)}")
    print(f"Missing from generated union:         {len(missing_from_generated)}")
    print(f"Extra in generated union:            {len(extra_in_generated)}")

    if missing_from_generated:
        print("First 20 missing IDs:")
        for x in sorted(missing_from_generated)[:20]:
            print(f"  {x}")

    if extra_in_generated:
        print("First 20 extra IDs:")
        for x in sorted(extra_in_generated)[:20]:
            print(f"  {x}")

    print_header("OFFICIAL TEST SPLIT PRESERVATION CHECK")
    test_missing = expected_test_pool - s_test
    test_extra = s_test - expected_test_pool

    print(f"Expected test IDs after intersection: {len(expected_test_pool)}")
    print(f"Generated test IDs:                   {len(s_test)}")
    print(f"Missing expected test IDs:            {len(test_missing)}")
    print(f"Extra test IDs not in official test:  {len(test_extra)}")

    if test_missing:
        print("First 20 missing official test IDs:")
        for x in sorted(test_missing)[:20]:
            print(f"  {x}")

    if test_extra:
        print("First 20 extra generated test IDs:")
        for x in sorted(test_extra)[:20]:
            print(f"  {x}")

    print_header("TRAIN/VAL SHOULD COME ONLY FROM OFFICIAL TRAIN POOL")
    non_train_pool_in_train = s_train - expected_train_pool
    non_train_pool_in_val = s_val - expected_train_pool

    print(f"Train IDs outside official train pool: {len(non_train_pool_in_train)}")
    print(f"Val IDs outside official train pool:   {len(non_train_pool_in_val)}")

    if non_train_pool_in_train:
        for x in sorted(non_train_pool_in_train)[:20]:
            print(f"  train outside pool: {x}")

    if non_train_pool_in_val:
        for x in sorted(non_train_pool_in_val)[:20]:
            print(f"  val outside pool: {x}")

    print_header("CAFE LEAKAGE CHECK")
    train_cafe2 = s_train & cafe2_ids
    val_cafe2 = s_val & cafe2_ids
    test_cafe2 = s_test & cafe2_ids

    print(f"Train IDs in cafe2: {len(train_cafe2)}")
    print(f"Val IDs in cafe2:   {len(val_cafe2)}")
    print(f"Test IDs in cafe2:  {len(test_cafe2)}")

    print_header("CSV CONSISTENCY CHECK")
    expected_union = s_train | s_val | s_test
    print(f"Rows in metadata CSV:          {len(csv_rows)}")
    print(f"Unique dish_ids in CSV:        {len(csv_id_set)}")
    print(f"Expected unique split dish_ids:{len(expected_union)}")
    print(f"Missing in CSV:                {len(expected_union - csv_id_set)}")
    print(f"Extra in CSV:                  {len(csv_id_set - expected_union)}")

    missing_csv = sorted(expected_union - csv_id_set)
    extra_csv = sorted(csv_id_set - expected_union)

    if missing_csv:
        print("First 20 missing CSV IDs:")
        for x in missing_csv[:20]:
            print(f"  {x}")

    if extra_csv:
        print("First 20 extra CSV IDs:")
        for x in extra_csv[:20]:
            print(f"  {x}")

    print_header("CSV SPLIT LABEL CHECK")
    csv_train = {row["dish_id"].strip() for row in csv_rows if row["split"].strip() == "train"}
    csv_val = {row["dish_id"].strip() for row in csv_rows if row["split"].strip() == "val"}
    csv_test = {row["dish_id"].strip() for row in csv_rows if row["split"].strip() == "test"}

    print(f"CSV train rows: {len(csv_train)}")
    print(f"CSV val rows:   {len(csv_val)}")
    print(f"CSV test rows:  {len(csv_test)}")

    print(f"CSV train mismatch vs txt: {len(csv_train ^ s_train)}")
    print(f"CSV val mismatch vs txt:   {len(csv_val ^ s_val)}")
    print(f"CSV test mismatch vs txt:  {len(csv_test ^ s_test)}")

    print_header("SOURCE_CAFE CHECK INSIDE CSV")
    if "source_cafe" in csv_rows[0]:
        source_counts = defaultdict(int)
        for row in csv_rows:
            source_counts[row["source_cafe"].strip()] += 1
        for k, v in sorted(source_counts.items()):
            print(f"{k}: {v}")
    else:
        print("No source_cafe column found in CSV.")

    print_header("TIMESTAMP SUMMARY")
    train_ts = summarize_timestamps(s_train)
    val_ts = summarize_timestamps(s_val)
    test_ts = summarize_timestamps(s_test)

    print(f"Train timestamps: {train_ts}")
    print(f"Val timestamps:   {val_ts}")
    print(f"Test timestamps:  {test_ts}")

    print_header("TIME-BUCKET LEAKAGE PROXY CHECK")
    train_buckets = {assign_time_bucket(x, BUCKET_SECONDS) for x in s_train}
    val_buckets = {assign_time_bucket(x, BUCKET_SECONDS) for x in s_val}
    test_buckets = {assign_time_bucket(x, BUCKET_SECONDS) for x in s_test}

    print(f"Train bucket count: {len(train_buckets)}")
    print(f"Val bucket count:   {len(val_buckets)}")
    print(f"Test bucket count:  {len(test_buckets)}")
    print()
    print(f"train ∩ val buckets  = {len(train_buckets & val_buckets)}")
    print(f"train ∩ test buckets = {len(train_buckets & test_buckets)}")
    print(f"val ∩ test buckets   = {len(val_buckets & test_buckets)}")

    print_header("FINAL PASS/FAIL SUMMARY")
    checks = {
        "No train/val overlap": len(s_train & s_val) == 0,
        "No train/test overlap": len(s_train & s_test) == 0,
        "No val/test overlap": len(s_val & s_test) == 0,
        "No duplicates in train file": len(train_ids) == len(s_train),
        "No duplicates in val file": len(val_ids) == len(s_val),
        "No duplicates in test file": len(test_ids) == len(s_test),
        "Generated union matches expected universe": len(missing_from_generated) == 0 and len(extra_in_generated) == 0,
        "Generated test matches official test pool": len(test_missing) == 0 and len(test_extra) == 0,
        "Train only from official train pool": len(non_train_pool_in_train) == 0,
        "Val only from official train pool": len(non_train_pool_in_val) == 0,
        "No cafe2 in train": len(train_cafe2) == 0,
        "No cafe2 in val": len(val_cafe2) == 0,
        "No cafe2 in test": len(test_cafe2) == 0,
        "CSV covers all split IDs": len(expected_union - csv_id_set) == 0,
        "CSV has no extra IDs": len(csv_id_set - expected_union) == 0,
        "CSV train matches txt": csv_train == s_train,
        "CSV val matches txt": csv_val == s_val,
        "CSV test matches txt": csv_test == s_test,
        "No train/val bucket overlap": len(train_buckets & val_buckets) == 0
    }

    for name, ok in checks.items():
        print(f"[{'PASS' if ok else 'FAIL'}] {name}")

if __name__ == "__main__":
    main()


BASIC SPLIT SUMMARY
Train IDs: 2421 (74.22%)
Val IDs:   334 (10.24%)
Test IDs:  507 (15.54%)
Total IDs: 3262

OVERLAP CHECKS
train ∩ val  = 0
train ∩ test = 0
val ∩ test   = 0

DUPLICATE CHECKS INSIDE EACH FILE
Duplicate train IDs: 0
Duplicate val IDs:   0
Duplicate test IDs:  0

EXPECTED SPLIT UNIVERSE CHECK
Expected official+available universe: 3262
Generated split union:                3262
Missing from generated union:         0
Extra in generated union:            0

OFFICIAL TEST SPLIT PRESERVATION CHECK
Expected test IDs after intersection: 507
Generated test IDs:                   507
Missing expected test IDs:            0
Extra test IDs not in official test:  0

TRAIN/VAL SHOULD COME ONLY FROM OFFICIAL TRAIN POOL
Train IDs outside official train pool: 0
Val IDs outside official train pool:   0

CAFE LEAKAGE CHECK
Train IDs in cafe2: 0
Val IDs in cafe2:   0
Test IDs in cafe2:  0

CSV CONSISTENCY CHECK
Rows in metadata CSV:          3262
Unique dish_ids in CSV:        3262
Exp

Creating new Metadata CSV for the dataset in order to incorporate the mass of each dish

In [4]:
import pandas as pd
from pathlib import Path
import csv

# =========================================================
# PATHS
# =========================================================
DATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset")
OGDATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\dish_ids")
OGDATASET_ROOT2 = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\metadata")

COMPACT_DIR = DATASET_ROOT / "nutrition5k_compactV3"
SPLITS_DIR = OGDATASET_ROOT / "splits"

DISH_METADATA_CAFE1 = OGDATASET_ROOT2 / "dish_metadata_cafe1.csv"
DISH_METADATA_CAFE2 = OGDATASET_ROOT2 / "dish_metadata_cafe2.csv"

OUTPUT_DIR = DATASET_ROOT / "prepared_overhead_rgb_dataset"

# Existing split metadata
SPLIT_METADATA_PATH = OUTPUT_DIR / "dish_carb_split_metadata.csv"

# New output metadata file
OUTPUT_METADATA_PATH = OUTPUT_DIR / "dish_carb_mass_split_metadata.csv"


# =========================================================
# LOAD DISH-LEVEL METADATA
# =========================================================
def load_single_dish_metadata(path: Path, source_cafe: str) -> pd.DataFrame:
    rows = []

    if not path.exists():
        raise FileNotFoundError(f"Metadata file not found: {path}")

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)

        for line_num, row in enumerate(reader, start=1):
            if not row:
                continue

            # Fixed leading Nutrition5K dish-level columns:
            # 0 = dish_id
            # 1 = total_calories
            # 2 = total_mass
            # 3 = total_fat
            # 4 = total_carb
            # 5 = total_protein
            if len(row) < 6:
                print(f"WARNING: Skipping short row in {path.name} at line {line_num}")
                continue

            dish_id = row[0].strip()

            try:
                total_mass = float(row[2])
                total_carb = float(row[4])
            except ValueError:
                print(f"WARNING: Skipping unparsable row in {path.name} at line {line_num}")
                continue

            rows.append({
                "dish_id": dish_id,
                "total_mass": total_mass,
                "total_carb_from_source": total_carb,
                "source_cafe": source_cafe
            })

    return pd.DataFrame(rows)


def load_dish_metadata() -> pd.DataFrame:
    cafe1_df = load_single_dish_metadata(DISH_METADATA_CAFE1, "cafe1")
    cafe2_df = load_single_dish_metadata(DISH_METADATA_CAFE2, "cafe2")
    return pd.concat([cafe1_df, cafe2_df], ignore_index=True)


# =========================================================
# LOAD EXISTING SPLIT METADATA
# Expected columns:
# dish_id, total_carb, split, source_cafe
# =========================================================
split_df = pd.read_csv(SPLIT_METADATA_PATH)

required_cols = {"dish_id", "total_carb", "split", "source_cafe"}
missing_cols = required_cols - set(split_df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns in split metadata: {missing_cols}")

print("Existing split metadata shape:", split_df.shape)
print(split_df.head())


# =========================================================
# LOAD NUTRITION5K DISH METADATA
# =========================================================
dish_meta_df = load_dish_metadata()

print("Dish-level Nutrition5K metadata shape:", dish_meta_df.shape)
print(dish_meta_df.head())


# =========================================================
# MERGE MASS INTO EXISTING SPLIT METADATA
# =========================================================
merged_df = split_df.merge(
    dish_meta_df,
    on=["dish_id", "source_cafe"],
    how="left"
)

# Optional consistency check against old carb column
carb_mismatch_mask = (merged_df["total_carb"] - merged_df["total_carb_from_source"]).abs() > 1e-6
num_mismatches = carb_mismatch_mask.fillna(False).sum()
print(f"Carb mismatches between existing metadata and source metadata: {num_mismatches}")

# Keep the original total_carb column from your current split metadata
# and only add total_mass
final_df = merged_df[["dish_id", "total_mass", "total_carb", "split", "source_cafe"]].copy()


# =========================================================
# CHECKS
# =========================================================
missing_mass = final_df["total_mass"].isna().sum()
print(f"Rows missing total_mass: {missing_mass}")

print("\nSplit counts:")
print(final_df["split"].value_counts(dropna=False))

print("\nPreview:")
print(final_df.head())


# =========================================================
# SAVE
# =========================================================
OUTPUT_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUTPUT_METADATA_PATH, index=False)

print(f"\nSaved new metadata file to:\n{OUTPUT_METADATA_PATH}")

Existing split metadata shape: (3262, 4)
           dish_id  total_carb split source_cafe
0  dish_1556575327      10.618  test       cafe1
1  dish_1557861216       0.000  test       cafe1
2  dish_1557862345       0.000  test       cafe1
3  dish_1557862696       3.850  test       cafe1
4  dish_1557862738       0.000  test       cafe1
Dish-level Nutrition5K metadata shape: (5006, 4)
           dish_id  total_mass  total_carb_from_source source_cafe
0  dish_1561662216       193.0               28.218290       cafe1
1  dish_1562688426        88.0                5.190000       cafe1
2  dish_1561662054       292.0               26.351543       cafe1
3  dish_1562008979       290.0               10.173570       cafe1
4  dish_1560455030       103.0                4.625000       cafe1
Carb mismatches between existing metadata and source metadata: 0
Rows missing total_mass: 0

Split counts:
split
train    2421
test      507
val       334
Name: count, dtype: int64

Preview:
           dish_id  tot

Adding ingredients of each dish to the meta data

In [1]:
import pandas as pd
from pathlib import Path
import csv
import re

# =========================================================
# PATHS
# =========================================================
DATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset")
OGDATASET_ROOT = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\dish_ids")
OGDATASET_ROOT2 = Path(r"C:\Users\jonas\OneDrive\Desktop\Speciale\Dataset\nutrition5k_dataset\nutrition5k_dataset\metadata")

COMPACT_DIR = DATASET_ROOT / "nutrition5k_compactV3"
SPLITS_DIR = OGDATASET_ROOT / "splits"

DISH_METADATA_CAFE1 = OGDATASET_ROOT2 / "dish_metadata_cafe1.csv"
DISH_METADATA_CAFE2 = OGDATASET_ROOT2 / "dish_metadata_cafe2.csv"

OUTPUT_DIR = DATASET_ROOT / "prepared_overhead_rgb_dataset"

SPLIT_METADATA_PATH = OUTPUT_DIR / "dish_carb_split_metadata.csv"

OUTPUT_METADATA_PATH = OUTPUT_DIR / "dish_carb_mass_ingredients_split_metadata.csv"


# =========================================================
# HELPERS
# =========================================================
def clean_ingredient_name(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def parse_single_dish_row(row, source_cafe: str):
    """
    Nutrition5K dish metadata format:
    0 = dish_id
    1 = total_calories
    2 = total_mass
    3 = total_fat
    4 = total_carb
    5 = total_protein

    Then repeating ingredient blocks of length 7:
    [ingredient_id, ingredient_name, ingredient_mass, ingredient_calories,
     ingredient_fat, ingredient_carb, ingredient_protein]
    """
    if len(row) < 6:
        return None

    dish_id = row[0].strip()

    try:
        total_mass = float(row[2])
        total_carb = float(row[4])
    except ValueError:
        return None

    ingredient_names = []
    ingredient_ids = []

    idx = 6
    block_size = 7

    while idx + block_size - 1 < len(row):
        ingr_id = row[idx].strip()
        ingr_name = row[idx + 1].strip()

        # Only store ingredient identity / presence
        # DO NOT store ingredient mass or nutrient values here
        if ingr_id and ingr_name:
            ingredient_ids.append(ingr_id)
            ingredient_names.append(clean_ingredient_name(ingr_name))

        idx += block_size

    ingredient_names = sorted(list(set(ingredient_names)))
    ingredient_ids = sorted(list(set(ingredient_ids)))

    return {
        "dish_id": dish_id,
        "total_mass": total_mass,
        "total_carb_from_source": total_carb,
        "source_cafe": source_cafe,
        "ingredient_names": ingredient_names,
        "ingredient_ids": ingredient_ids
    }


def load_single_dish_metadata(path: Path, source_cafe: str) -> pd.DataFrame:
    rows = []

    if not path.exists():
        raise FileNotFoundError(f"Metadata file not found: {path}")

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)

        for line_num, row in enumerate(reader, start=1):
            if not row:
                continue

            parsed = parse_single_dish_row(row, source_cafe)
            if parsed is None:
                print(f"WARNING: Skipping invalid row in {path.name} at line {line_num}")
                continue

            rows.append(parsed)

    return pd.DataFrame(rows)


def load_dish_metadata() -> pd.DataFrame:
    cafe1_df = load_single_dish_metadata(DISH_METADATA_CAFE1, "cafe1")
    cafe2_df = load_single_dish_metadata(DISH_METADATA_CAFE2, "cafe2")
    return pd.concat([cafe1_df, cafe2_df], ignore_index=True)


# =========================================================
# LOAD EXISTING SPLIT METADATA
# Expected columns:
# dish_id, total_carb, split, source_cafe
# =========================================================
split_df = pd.read_csv(SPLIT_METADATA_PATH)

required_cols = {"dish_id", "total_carb", "split", "source_cafe"}
missing_cols = required_cols - set(split_df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns in split metadata: {missing_cols}")

print("Existing split metadata shape:", split_df.shape)
print(split_df.head())


# =========================================================
# LOAD NUTRITION5K DISH METADATA
# =========================================================
dish_meta_df = load_dish_metadata()

print("Dish-level Nutrition5K metadata shape:", dish_meta_df.shape)
print(dish_meta_df.head())


# =========================================================
# MERGE MASS + INGREDIENT LISTS INTO EXISTING SPLIT METADATA
# =========================================================
merged_df = split_df.merge(
    dish_meta_df,
    on=["dish_id", "source_cafe"],
    how="left"
)

# Optional consistency check against old carb column
carb_mismatch_mask = (merged_df["total_carb"] - merged_df["total_carb_from_source"]).abs() > 1e-6
num_mismatches = carb_mismatch_mask.fillna(False).sum()
print(f"Carb mismatches between existing metadata and source metadata: {num_mismatches}")


# =========================================================
# BUILD MULTI-HOT INGREDIENT COLUMNS
# =========================================================
all_ingredients = set()

for ingr_list in merged_df["ingredient_names"].dropna():
    for ingr in ingr_list:
        all_ingredients.add(ingr)

all_ingredients = sorted(all_ingredients)
print(f"Total unique ingredients: {len(all_ingredients)}")

for ingr in all_ingredients:
    col_name = f"ingr_{ingr}"
    merged_df[col_name] = merged_df["ingredient_names"].apply(
        lambda x: int(ingr in x) if isinstance(x, list) else 0
    )


# =========================================================
# OPTIONAL EXTRA SUMMARY FEATURES
# =========================================================
merged_df["num_ingredients"] = merged_df["ingredient_names"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

merged_df["ingredient_list_str"] = merged_df["ingredient_names"].apply(
    lambda x: "|".join(x) if isinstance(x, list) else ""
)


# =========================================================
# FINAL DATAFRAME
# Keep one dish per row
# =========================================================
ingredient_cols = [f"ingr_{ingr}" for ingr in all_ingredients]

final_cols = [
    "dish_id",
    "total_mass",
    "total_carb",
    "split",
    "source_cafe",
    "num_ingredients",
    "ingredient_list_str"
] + ingredient_cols

final_df = merged_df[final_cols].copy()


# =========================================================
# CHECKS
# =========================================================
missing_mass = final_df["total_mass"].isna().sum()
print(f"Rows missing total_mass: {missing_mass}")

missing_ingr_lists = (final_df["ingredient_list_str"] == "").sum()
print(f"Rows missing ingredient lists: {missing_ingr_lists}")

print("\nSplit counts:")
print(final_df["split"].value_counts(dropna=False))

print("\nPreview:")
print(final_df.iloc[:, :15].head())


# =========================================================
# SAVE
# =========================================================
OUTPUT_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUTPUT_METADATA_PATH, index=False)

print(f"\nSaved new metadata file to:\n{OUTPUT_METADATA_PATH}")

Existing split metadata shape: (3262, 4)
           dish_id  total_carb split source_cafe
0  dish_1556575327      10.618  test       cafe1
1  dish_1557861216       0.000  test       cafe1
2  dish_1557862345       0.000  test       cafe1
3  dish_1557862696       3.850  test       cafe1
4  dish_1557862738       0.000  test       cafe1
Dish-level Nutrition5K metadata shape: (5006, 6)
           dish_id  total_mass  total_carb_from_source source_cafe  \
0  dish_1561662216       193.0               28.218290       cafe1   
1  dish_1562688426        88.0                5.190000       cafe1   
2  dish_1561662054       292.0               26.351543       cafe1   
3  dish_1562008979       290.0               10.173570       cafe1   
4  dish_1560455030       103.0                4.625000       cafe1   

                                    ingredient_names  \
0  [apple, bok_choy, brown_rice, garlic, lemon_ju...   
1          [chicken_apple_sausage, roasted_potatoes]   
2  [apple, bok_choy, garlic

C:\Users\jonas\AppData\Local\Temp\ipykernel_27056\2703156524.py:175: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col_name] = merged_df["ingredient_names"].apply(
C:\Users\jonas\AppData\Local\Temp\ipykernel_27056\2703156524.py:175: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col_name] = merged_df["ingredient_names"].apply(
C:\Users\jonas\AppData\Local\Temp\ipykernel_27056\2703156524.py:175: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times,

Rows missing total_mass: 0
Rows missing ingredient lists: 0

Split counts:
split
train    2421
test      507
val       334
Name: count, dtype: int64

Preview:
           dish_id  total_mass  total_carb split source_cafe  num_ingredients  \
0  dish_1556575327       152.0      10.618  test       cafe1                3   
1  dish_1557861216         1.0       0.000  test       cafe1                1   
2  dish_1557862345        63.0       0.000  test       cafe1                1   
3  dish_1557862696        77.0       3.850  test       cafe1                1   
4  dish_1557862738       118.0       0.000  test       cafe1                1   

              ingredient_list_str  ingr_almonds  ingr_apple  ingr_artichokes  \
0  brussels_sprouts|celery|olives             0           0                0   
1                      plate_only             0           0                0   
2                         chicken             0           0                0   
3                     cauliflower 